# Solved Examples: Reward Shaping & Hyperparameter Effects

**Solved Examples Notebook — Weeks 2 & 3**

This notebook covers two core skills on simple, fast environments before you apply them to Walker2D:

- **Part A (Reward Shaping):** How different reward functions change agent behaviour, using a 5×5 GridWorld.
- **Part B (Hyperparameter Effects):** How key PPO hyperparameters affect learning, using CartPole-v1.

Both environments train in seconds, so you can see every effect clearly.

---
**All code is solved and explained. Run cells in order and read the commentary.**

---
# Part A: Reward Shaping in GridWorld

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
from collections import defaultdict

np.random.seed(0)
random.seed(0)

# ---- GridWorld constants ----
GRID_SIZE   = 5
N_STATES    = 25
N_ACTIONS   = 4
GOAL_STATE  = 24
PIT_STATE   = 17
GAMMA       = 0.9
ACTIONS     = {0:(-1,0), 1:(1,0), 2:(0,-1), 3:(0,1)}
ACTION_NAMES= ['Up','Down','Left','Right']

def state_to_rc(s): return s // GRID_SIZE, s % GRID_SIZE
def rc_to_state(r,c): return r*GRID_SIZE+c

def transition(state, action):
    """Deterministic transition. Returns next_state."""
    if state in (GOAL_STATE, PIT_STATE): return state
    r, c = state_to_rc(state)
    dr, dc = ACTIONS[action]
    nr, nc = int(np.clip(r+dr,0,4)), int(np.clip(c+dc,0,4))
    return rc_to_state(nr, nc)

def is_terminal(s): return s in (GOAL_STATE, PIT_STATE)

print('GridWorld setup OK.')

## A1: Define Four Reward Functions

| Name | Description | Walker2D Analogy |
|------|-------------|------------------|
| `sparse` | +10 at goal, −10 at pit, 0 elsewhere | Only reward on success/failure |
| `dense_step` | −1 per step + terminal bonuses | Alive bonus + velocity reward |
| `shaped_distance` | Reward ∝ closeness to goal | Velocity reward (progress proxy) |
| `misleading` | Large bonus near the pit | Poorly designed reward → reward hacking |

In [ ]:
def manhattan_to_goal(state):
    r, c = state_to_rc(state)
    gr, gc = state_to_rc(GOAL_STATE)
    return abs(r - gr) + abs(c - gc)


def reward_sparse(state, action, next_state):
    """Only signals at goal and pit."""
    if next_state == GOAL_STATE: return +10.0
    if next_state == PIT_STATE:  return -10.0
    return 0.0


def reward_dense_step(state, action, next_state):
    """Small cost per step plus terminal signals."""
    if next_state == GOAL_STATE: return +10.0
    if next_state == PIT_STATE:  return -10.0
    return -1.0


def reward_shaped_distance(state, action, next_state):
    """
    Potential-based shaping: reward = reduction in distance to goal.
    Theoretically grounded — does not change the optimal policy.
    """
    if next_state == GOAL_STATE: return +10.0
    if next_state == PIT_STATE:  return -10.0
    d_before = manhattan_to_goal(state)
    d_after  = manhattan_to_goal(next_state)
    return float(d_before - d_after) - 0.1


def reward_misleading(state, action, next_state):
    """
    Deliberately bad reward: large bonus near the pit.
    Models a 'reward hack' where proxy reward diverges from true goal.
    """
    if next_state == GOAL_STATE: return +10.0
    if next_state == PIT_STATE:  return -10.0
    r, c = state_to_rc(next_state)
    pr, pc = state_to_rc(PIT_STATE)
    dist_to_pit = abs(r - pr) + abs(c - pc)
    if dist_to_pit == 1:
        return +5.0
    return -1.0


REWARD_FNS = {
    'sparse':           reward_sparse,
    'dense_step':       reward_dense_step,
    'shaped_distance':  reward_shaped_distance,
    'misleading':       reward_misleading,
}

print('Reward functions defined.')

## A2: Q-Learning with Each Reward Function

In [ ]:
def q_learning_with_reward(reward_fn, n_episodes=5000, alpha=0.1, gamma=GAMMA,
                            eps_start=1.0, eps_end=0.05, eps_decay=0.999,
                            seed=0):
    """Train Q-Learning on GridWorld using the given reward function."""
    random.seed(seed); np.random.seed(seed)

    Q  = np.zeros((N_STATES, N_ACTIONS))
    eps = eps_start
    ep_rewards  = []
    ep_successes = []

    for ep in range(n_episodes):
        state = 0
        total_r = 0.0
        success = False

        for _ in range(200):
            if random.random() < eps:
                action = random.randint(0, N_ACTIONS-1)
            else:
                action = int(np.argmax(Q[state]))

            next_state = transition(state, action)
            reward     = reward_fn(state, action, next_state)
            done       = is_terminal(next_state)
            total_r   += reward

            td_target = reward + gamma * np.max(Q[next_state])
            Q[state, action] += alpha * (td_target - Q[state, action])

            state = next_state
            if done:
                if next_state == GOAL_STATE: success = True
                break

        ep_rewards.append(total_r)
        ep_successes.append(float(success))
        eps = max(eps_end, eps * eps_decay)

    policy = np.argmax(Q, axis=1)
    return Q, policy, ep_rewards, ep_successes


# Train all four
results = {}
for name, rfn in REWARD_FNS.items():
    print(f'Training with {name} reward...')
    Q, pol, rews, succs = q_learning_with_reward(rfn)
    results[name] = dict(Q=Q, policy=pol, rewards=rews, successes=succs)

print('\nSuccess rates (last 500 episodes):')
for name, res in results.items():
    sr = np.mean(res['successes'][-500:]) * 100
    print(f'  {name:20s}  {sr:.1f}%')

In [ ]:
# Plot success rate over time
window = 200
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['steelblue', 'seagreen', 'darkorange', 'tomato']

for (name, res), color in zip(results.items(), colors):
    smoothed = np.convolve(res['successes'], np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(res['successes'])),
            smoothed * 100, label=name, color=color, linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Success rate (%, rolling average)')
ax.set_title('Reward Function Comparison: Success Rate on GridWorld')
ax.legend()
ax.grid(True, alpha=0.4)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

**What you should observe:**

- **`sparse`**: learns slowly or not at all. The signal is too weak for the agent to discover the goal.
- **`dense_step`**: learns steadily and eventually reaches high success rate.
- **`shaped_distance`**: learns fastest because every step toward the goal is immediately rewarded.
- **`misleading`**: may learn to exploit the pit-adjacent bonus instead of reaching the goal — **reward hacking**.

## A3: Visualise the Learned Policies

In [ ]:
def plot_policy_only(policy, title, ax):
    colors_map = {GOAL_STATE: 'limegreen', PIT_STATE: 'tomato'}
    for s in range(N_STATES):
        r, c = state_to_rc(s)
        y = GRID_SIZE - 1 - r
        fc = colors_map.get(s, 'white')
        rect = mpatches.FancyBboxPatch((c,y),1,1,
                   boxstyle='square,pad=0', linewidth=1,
                   edgecolor='black', facecolor=fc)
        ax.add_patch(rect)
        lbl = 'G' if s==GOAL_STATE else ('P' if s==PIT_STATE else '')
        ax.text(c+0.5, y+0.65, lbl, ha='center', va='center', fontsize=9, fontweight='bold')
        if s not in (GOAL_STATE, PIT_STATE):
            dr, dc = ACTIONS[policy[s]]
            ax.annotate('', xy=(c+0.5+dc*0.3, y+0.5-dr*0.3),
                        xytext=(c+0.5, y+0.5),
                        arrowprops=dict(arrowstyle='->', color='navy', lw=1.5))
    ax.set_xlim(0,GRID_SIZE); ax.set_ylim(0,GRID_SIZE)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=11)


fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    plot_policy_only(res['policy'], name, ax)
plt.suptitle('Learned Policies for Each Reward Function', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Look at the `misleading` policy carefully.**  
Do the arrows near the pit point *toward* the pit or *toward* the goal?  
This is a miniature version of reward hacking: the agent learned to maximise the proxy reward (bonus near pit) rather than the true goal.

In Walker2D, the same thing can happen:  
if the alive bonus ($c_2$) is too large, the agent learns to stand still forever instead of walking.

## A4: How Coefficient Magnitude Affects Learning

We test how changing the step penalty magnitude changes the optimal path.  
This mirrors changing $c_3$ (energy penalty) in the Walker reward.

In [ ]:
def reward_dense_with_penalty(penalty):
    """Factory that creates a dense reward with a given per-step penalty."""
    def rfn(state, action, next_state):
        if next_state == GOAL_STATE: return +10.0
        if next_state == PIT_STATE:  return -10.0
        return -penalty
    rfn.__name__ = f'dense_p{penalty}'
    return rfn


penalties  = [0.0, 0.5, 1.0, 3.0]
pen_results = {}

for p in penalties:
    print(f'Training with step penalty = {p} ...')
    _, pol, rews, succs = q_learning_with_reward(reward_dense_with_penalty(p), n_episodes=3000)
    pen_results[p] = dict(policy=pol, successes=succs)

print('\nSuccess rates (last 500 eps):')
for p, res in pen_results.items():
    sr = np.mean(res['successes'][-500:]) * 100
    print(f'  step_penalty={p:.1f}   success={sr:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (p, res) in zip(axes, pen_results.items()):
    plot_policy_only(res['policy'], f'step_penalty = {p}', ax)
plt.suptitle('Effect of Step Penalty Coefficient on Learned Policy', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Key insight:**

- `penalty = 0.0` (no step cost): agent can reach the goal along *any* path — no incentive to be efficient.
- `penalty = 1.0` (moderate): agent finds the shortest path.
- `penalty = 3.0` (large): if the penalty is large enough, the agent decides the goal isn't worth the cost and gives up.

This maps directly to Walker2D:
- $c_3 = 0$ (velocity-only reward): agent has no energy incentive, may use huge torques.
- $c_3 = 0.001$ (default): balanced trade-off.
- $c_3 = 0.01$ (heavy energy penalty): agent stays still to avoid any penalty.

## A5: Summary Table

In [ ]:
print(f'{"Reward Function":<22} {"Success % (last 500 eps)":>28}')
print('-' * 52)
for name, res in results.items():
    sr = np.mean(res['successes'][-500:]) * 100
    print(f'{name:<22} {sr:>28.1f}%')
print()
print('Note: success = reached goal state (state 24) before falling in pit.')

## A6: Reflection Questions (Reward Shaping)

**Q1.** The `shaped_distance` reward usually learns faster than `dense_step`.  
Why is giving feedback on *progress toward the goal* better than a fixed step cost?

**Q2.** Look at the `misleading` policy. What strategy did the agent learn?  
How would you fix the reward function to prevent this?

**Q3.** In the Walker2D dense reward, velocity is clipped: `clip(vx, 0, 5)`.  
Why is it clipped at 5? What would happen with no upper bound?

**Q4.** The alive bonus ($c_2 = 1.0$) is always $+1$ as long as the robot hasn't fallen.  
If you doubled it to $c_2 = 2.0$, what behaviour might develop?  
What if $c_1 = 0$ (no velocity reward)?

**Q5.** Potential-based reward shaping (the `shaped_distance` function) is theoretically guaranteed not to change the optimal policy.  
Does the Walker2D dense reward use potential-based shaping? Explain why or why not.

---
---
# Part B: Hyperparameter Effects on CartPole

In [ ]:
import gymnasium as gym
import os

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

BASE_LOG  = './tb_examples'
BASE_SAVE = './models_examples'
os.makedirs(BASE_LOG, exist_ok=True)
os.makedirs(BASE_SAVE, exist_ok=True)


def run_experiment(name, total_steps=80_000, **ppo_kwargs):
    """
    Train PPO on CartPole-v1 with the given kwargs and return the eval timeseries.
    """
    save_dir = os.path.join(BASE_SAVE, name)
    os.makedirs(save_dir, exist_ok=True)

    venv = make_vec_env('CartPole-v1', n_envs=4, seed=0)
    defaults = dict(
        policy='MlpPolicy', env=venv,
        learning_rate=3e-4, n_steps=2048, batch_size=64,
        n_epochs=10, gamma=0.99, gae_lambda=0.95,
        clip_range=0.2, ent_coef=0.0, verbose=0,
        tensorboard_log=BASE_LOG,
    )
    defaults.update(ppo_kwargs)
    model = PPO(**defaults)

    ev = make_vec_env('CartPole-v1', n_envs=1, seed=99)
    cb = EvalCallback(ev, best_model_save_path=save_dir, log_path=save_dir,
                      eval_freq=5000, n_eval_episodes=20,
                      deterministic=True, verbose=0)

    model.learn(total_timesteps=total_steps, callback=cb,
                tb_log_name=name, reset_num_timesteps=True)
    ev.close(); venv.close()

    data = np.load(os.path.join(save_dir, 'evaluations.npz'))
    return data['timesteps'], data['results'].mean(axis=1), data['results'].std(axis=1)


def plot_comparison(experiments, title, ax=None):
    """
    experiments: list of (label, timesteps, mean_rewards, std_rewards)
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(9, 4))
    colors = plt.cm.tab10(np.linspace(0, 1, len(experiments)))
    for (label, ts, means, stds), color in zip(experiments, colors):
        ax.fill_between(ts, means-stds, means+stds, alpha=0.2, color=color)
        ax.plot(ts, means, label=label, color=color, linewidth=2)
    ax.axhline(500, color='green', linestyle='--', linewidth=1, label='Max')
    ax.set_xlabel('Timesteps'); ax.set_ylabel('Mean reward')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.4)
    if standalone:
        plt.tight_layout(); plt.show()


print('Setup OK.')

## B1: Learning Rate

The learning rate $\alpha$ is often the most important hyperparameter.  
We test three values: $10^{-4}$ (too small), $3\times10^{-4}$ (good default), $10^{-2}$ (too large).

In [ ]:
print('Experiment 1: Learning rate comparison')
print('Running lr=1e-4 ...')
lr_slow   = run_experiment('lr_slow',   learning_rate=1e-4)
print('Running lr=3e-4 (default) ...')
lr_good   = run_experiment('lr_good',   learning_rate=3e-4)
print('Running lr=1e-2 (too large) ...')
lr_large  = run_experiment('lr_large',  learning_rate=1e-2)
print('Done.')

In [ ]:
plot_comparison([
    ('lr=1e-4 (too small)',  *lr_slow),
    ('lr=3e-4 (default)',    *lr_good),
    ('lr=1e-2 (too large)',  *lr_large),
], 'Learning Rate Comparison on CartPole-v1')

**Expected pattern:**
- `lr=1e-4`: Learns slowly but stably. May not reach 500 within 80k steps.
- `lr=3e-4`: Reaches high reward and stays there.
- `lr=1e-2`: Learns fast initially but then collapses or oscillates — updates are too large.

**Rule of thumb for PPO:** Start at $3\times10^{-4}$.  
If training is unstable (`approx_kl` exceeds 0.05), lower it.  
If training is stable but slow, try increasing it slightly.

## B2: Network Architecture

In [ ]:
print('Experiment 2: Network architecture')
print('Running net=32x32 ...')
net_tiny   = run_experiment('net_tiny',   policy_kwargs=dict(net_arch=[32, 32]))
print('Running net=64x64 ...')
net_small  = run_experiment('net_small',  policy_kwargs=dict(net_arch=[64, 64]))
print('Running net=256x256 (default) ...')
net_medium = run_experiment('net_medium', policy_kwargs=dict(net_arch=[256, 256]))
print('Running net=512x512 ...')
net_large  = run_experiment('net_large',  policy_kwargs=dict(net_arch=[512, 512]))
print('Done.')

In [ ]:
plot_comparison([
    ('[32, 32]',   *net_tiny),
    ('[64, 64]',   *net_small),
    ('[256, 256]', *net_medium),
    ('[512, 512]', *net_large),
], 'Network Architecture Comparison on CartPole-v1')

**Expected pattern:**  
CartPole is simple (4-dim observation, 2 actions) — even a 32×32 network should solve it.  
Larger networks may converge at a similar speed or slightly slower.

**For Walker2D** (22-dim observation, 6 continuous actions), a [64, 64] network may not have enough capacity.  
[256, 256] is a reasonable default. [512, 512] usually isn't worth the extra training time.

## B3: Rollout Length (`n_steps`)

`n_steps` controls how many transitions are collected before each PPO update.  
With 4 parallel envs and `n_steps=2048`, we collect $4 \times 2048 = 8192$ transitions per update.

In [ ]:
print('Experiment 3: Rollout length (n_steps)')
print('Running n_steps=256 ...')
ns_256  = run_experiment('ns_256',  n_steps=256,  batch_size=32)
print('Running n_steps=1024 ...')
ns_1024 = run_experiment('ns_1024', n_steps=1024, batch_size=64)
print('Running n_steps=2048 (default) ...')
ns_2048 = run_experiment('ns_2048', n_steps=2048, batch_size=64)
print('Running n_steps=4096 ...')
ns_4096 = run_experiment('ns_4096', n_steps=4096, batch_size=128)
print('Done.')

In [ ]:
plot_comparison([
    ('n_steps=256  (short)', *ns_256),
    ('n_steps=1024',         *ns_1024),
    ('n_steps=2048 (default)',*ns_2048),
    ('n_steps=4096 (long)',  *ns_4096),
], 'Rollout Length (n_steps) Comparison on CartPole-v1')

**Expected pattern:**  
Short rollout (256): many updates, but each uses noisy advantage estimates.  
Long rollout (4096): fewer updates, but each estimate is smoother.

For CartPole, differences are small.  
For Walker2D, longer rollouts tend to work better because the value function needs more data for accurate advantage estimates.

## B4: Discount Factor (gamma)

In [ ]:
print('Experiment 4: Discount factor (gamma)')
print('Running gamma=0.9 ...')
g_90  = run_experiment('g_90',  gamma=0.90)
print('Running gamma=0.95 ...')
g_95  = run_experiment('g_95',  gamma=0.95)
print('Running gamma=0.99 (default) ...')
g_99  = run_experiment('g_99',  gamma=0.99)
print('Running gamma=0.999 ...')
g_999 = run_experiment('g_999', gamma=0.999)
print('Done.')

In [ ]:
plot_comparison([
    ('gamma=0.90',  *g_90),
    ('gamma=0.95',  *g_95),
    ('gamma=0.99 (default)', *g_99),
    ('gamma=0.999', *g_999),
], 'Discount Factor (gamma) Comparison on CartPole-v1')

**Expected pattern:**

CartPole gives $+1$ per step and terminates at step 500.  
With $\gamma = 0.9$, the value of step 100 is $0.9^{100} \approx 2.6\times10^{-5}$: almost zero.  
The agent won't plan far enough ahead to sustain balance.

With $\gamma = 0.99$, step 100 contributes $0.99^{100} \approx 0.37$: still significant.

For Walker2D, high $\gamma$ (0.99) is important because a mistake now may only cause a fall 50–200 steps later.

## B5: All Experiments on One Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plot_comparison([
    ('lr=1e-4', *lr_slow), ('lr=3e-4 (default)', *lr_good), ('lr=1e-2', *lr_large)
], 'Learning Rate', axes[0, 0])

plot_comparison([
    ('[32,32]', *net_tiny), ('[64,64]', *net_small),
    ('[256,256]', *net_medium), ('[512,512]', *net_large)
], 'Network Architecture', axes[0, 1])

plot_comparison([
    ('n_steps=256', *ns_256), ('n_steps=1024', *ns_1024),
    ('n_steps=2048', *ns_2048), ('n_steps=4096', *ns_4096)
], 'Rollout Length (n_steps)', axes[1, 0])

plot_comparison([
    ('gamma=0.90', *g_90), ('gamma=0.95', *g_95),
    ('gamma=0.99', *g_99), ('gamma=0.999', *g_999)
], 'Discount Factor (gamma)', axes[1, 1])

plt.suptitle('PPO Hyperparameter Effects on CartPole-v1', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('week2_3_hyperparameter_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved overview plot.')

## B6: Summary Table

In [ ]:
experiments_all = [
    ('lr=1e-4',          lr_slow  [1][-1], lr_slow  [2][-1]),
    ('lr=3e-4 (default)',lr_good  [1][-1], lr_good  [2][-1]),
    ('lr=1e-2',          lr_large [1][-1], lr_large [2][-1]),
    ('[32,32]',          net_tiny [1][-1], net_tiny [2][-1]),
    ('[64,64]',          net_small[1][-1], net_small[2][-1]),
    ('[256,256]',        net_medium[1][-1],net_medium[2][-1]),
    ('[512,512]',        net_large[1][-1], net_large[2][-1]),
    ('n_steps=256',      ns_256   [1][-1], ns_256   [2][-1]),
    ('n_steps=2048',     ns_2048  [1][-1], ns_2048  [2][-1]),
    ('n_steps=4096',     ns_4096  [1][-1], ns_4096  [2][-1]),
    ('gamma=0.90',       g_90     [1][-1], g_90     [2][-1]),
    ('gamma=0.99',       g_99     [1][-1], g_99     [2][-1]),
    ('gamma=0.999',      g_999    [1][-1], g_999    [2][-1]),
]

print(f'{"Config":<25} {"Final Mean":>12} {"Std":>8}')
print('-' * 48)
for name, mean, std in sorted(experiments_all, key=lambda x: -x[1]):
    print(f'{name:<25} {mean:>12.1f} {std:>8.1f}')

## B7: Reflection Questions (Hyperparameters)

**Q1.** In the learning rate plot, why does a too-large LR cause performance to rise quickly at first but then fall?  
(Think about what happens to the policy gradient step when $\alpha$ is large.)

**Q2.** For CartPole, even a [32, 32] network solves the problem.  
Why might this not hold for Walker2D? What is different about the task?

**Q3.** `n_steps` controls the trade-off between update frequency and gradient variance.  
If you double `n_steps` from 2048 to 4096, how does that change the number of gradient updates per 100k environment steps?

**Q4.** Compute $\gamma^{100}$ for $\gamma \in \{0.9, 0.95, 0.99, 0.999\}$.  
For which values does the agent still "care about" what happens 100 steps in the future?  
Why does this matter for locomotion?

**Q5.** Based on these CartPole experiments, which hyperparameter do you expect to matter *most* for Walker2D?  
How would you design your ablation study to verify this?